# Local Testing Notebook

This notebook runs the experiments locally using the current working directory, so you don't need to push to GitHub or use Google Colab.

In [ ]:
import sys
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Image, Markdown

# Set the project root to the directory containing this notebook.
# The mmdp package must be installed first: pip install -e .[notebooks]
PROJECT_ROOT = Path('.').resolve()

OUTPUT_DIR = PROJECT_ROOT / 'local_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / 'MMDP_results.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CSV:', RESULTS_CSV)

In [ ]:
from mmdp.notebook_visualizer import plot_map_visualization
from ipywidgets import interact, IntSlider

In [ ]:
GROUP_LABELS = {'easy':'Easy', 'medium':'Medium', 'hard':'Hard'}
GROUP_MAPS = {
    'easy': ['empty-8-8'],
    'medium': ['warehouse-10-20-10-2-1'],
    'hard': ['room-64-64-16'],
}

def run_group(group):
    print('='*80)
    print(f"Starting group {GROUP_LABELS[group]}: {', '.join(GROUP_MAPS[group])}")
    print('='*80, flush=True)
    command = [
        sys.executable, '-u', str(PROJECT_ROOT / 'scripts' / 'run_compact_matrix.py'),
        '--group', group, '--output', str(RESULTS_CSV),
    ]
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'Experiment failed with code {code}')

def comparison_table(group):
    df = pd.read_csv(RESULTS_CSV)
    part = df[(df['map_group'] == group) & (df['status'] == 'ok')].copy()
    metrics = ['planning_time_seconds','planning_peak_memory_delta_mb','states_examined','success_rate']
    for col in ['n_agents', *metrics]:
        part[col] = pd.to_numeric(part[col], errors='coerce')
    means = part.groupby(['n_agents','algorithm'], as_index=False)[metrics].mean()
    wide = means.pivot(index='n_agents', columns='algorithm', values=metrics)
    result = pd.DataFrame(index=wide.index)
    names = {
        'planning_time_seconds':'time_s',
        'planning_peak_memory_delta_mb':'memory_mb',
        'states_examined':'states',
        'success_rate':'success',
    }
    for metric, short in names.items():
        if (metric, 'baseline') in wide and (metric, 'od') in wide:
            result[f'{short}_baseline'] = wide[(metric,'baseline')]
            result[f'{short}_od'] = wide[(metric,'od')]
            result[f'{short}_diff_OD_minus_Base'] = wide[(metric,'od')] - wide[(metric,'baseline')]
    return result.reset_index().round(4)

def analyze_group(group):
    graph_dir = OUTPUT_DIR / 'graphs' / group
    command = [
        sys.executable, str(PROJECT_ROOT / 'scripts' / 'analyze_compact_results.py'),
        str(RESULTS_CSV), '--group', group, '--output-dir', str(graph_dir),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    display(Markdown('## Average Comparison by Agents'))
    display(comparison_table(group))
    figures = [
        ('01_time.png', 'Planning Time (Lower is better)'),
        ('02_memory.png', 'Memory Delta (Lower is better)'),
        ('03_states_and_success.png', 'States Examined & Success Rate'),
    ]
    for filename, title in figures:
        fig_path = graph_dir / filename
        if fig_path.exists():
            display(Markdown(f'### {title}'))
            display(Image(filename=str(fig_path)))

def run_and_analyze(group):
    run_group(group)
    analyze_group(group)

## Visualizing Maps

In [ ]:
def _plot_easy(num_agents):
    plot_map_visualization('empty-8-8', 'empty-8-8-bundled-1', num_agents)
interact(_plot_easy, num_agents=IntSlider(min=1, max=6, step=1, value=3, description='Agents:'))

## Running Easy Experiment

In [ ]:
run_and_analyze("easy")

## Running Medium Experiment

In [ ]:
run_and_analyze("medium")

## Running Hard Experiment

In [ ]:
run_and_analyze("hard")